<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_41_api_testing/note_lesson_41_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Урок 41 — Тестування API: новинний агрегатор, крок 5

| Крок | Що робимо |
|---|---|
| 0 | `news_hub` з уроку 39 з перебудованими тестами; pytest з ноутбука |
| 1 | шари тестів і маркери (вправа 1) |
| 2 | збережені сторінки й тест конвеєра (вправи 2–3) |
| 3 | мок мережі: patch where used (вправи 4–6) |
| 4 | фейковий HTTP-сервер (вправа 7) |
| 5 | API асинхронно: `httpx.AsyncClient` (вправа 8) |
| 6 | покриття (вправа 9) |
| 7 | знайди помилку: «майже rbc.ua» (вправа 10) |

Серверів не треба: база — SQLite, Redis — `fakeredis`, «сайт» — локальний `aiohttp`-сервер. Теорія й розбір знахідок — на [сторінці уроку](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_41/).

---
## 0. Проєкт і pytest

Клітинка завантажує `news_hub` (у Colab), ставить залежності й переходить у папку проєкту. `run_pytest(...)` запускає pytest окремим процесом — так само, як у терміналі — і повертає вивід.

У ноутбуці можна писати `await` прямо в клітинці: Jupyter уже має запущений цикл подій (тому `asyncio.run(...)` тут не працює).

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_41_api_testing"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, fakeredis, fastapi, httpx, pytest_asyncio, pytest_cov  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///lesson41.db"
os.environ["REDIS_URL"] = "fakeredis://"


def run_pytest(*args: str) -> str:
    """pytest окремим процесом (як у терміналі); повертає вивід."""
    env = {key: value for key, value in os.environ.items() if key not in ("DATABASE_URL", "REDIS_URL")}
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", *args],
                            capture_output=True, text=True, env=env)
    return result.stdout + result.stderr


print(run_pytest().splitlines()[-1])

---
## 1. Шари тестів

`tests/unit/` — одна функція чи клас без бази, Redis і мережі. `tests/integration/` — застосунок цілком. Маркер `unit` / `integration` ставить `tests/conftest.py` за назвою папки.

### Вправа 1

Запусти лише unit-тести (`-m unit`) і лише інтеграційні. Збережи **останні рядки** виводу в `unit_line` і `integration_line`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
unit_line = run_pytest("-m", "unit").splitlines()[-1]
integration_line = run_pytest("-m", "integration").splitlines()[-1]
# END SOLUTION

print(unit_line)
print(integration_line)
assert "passed" in unit_line and "deselected" in unit_line
assert "passed" in integration_line and "failed" not in integration_line
print("✅ Вправа 1 пройдена")

---
## 2. Збережені сторінки

`tests/fixtures/rbc_newsline_item.html` — справжній фрагмент стрічки rbc.ua, `demo_newsline.html` — демо-розмітка з контейнерами `newsline__item`.

### Вправа 2

Прочитай `tests/fixtures/rbc_newsline_item.html` (UTF-8), розбери `parse_rbc_news` і збережи список у `news`.

In [ ]:
from pathlib import Path

from news_hub.models import NewsItem, validate_news
from news_hub.parser import parse_rbc_news

# YOUR CODE HERE
# BEGIN SOLUTION
news = parse_rbc_news(Path("tests/fixtures/rbc_newsline_item.html").read_text(encoding="utf-8"))
# END SOLUTION

print(news)
assert len(news) == 1 and news[0]["datetime"] == "02:06"
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Парсер з `demo_newsline.html` віддає `datetime = "2024-05-08T10:30:00"` (з атрибута `<time datetime>`). У моделі `NewsItem` поле `published_time: time | None`. Чи прийме його модель уроку 39, де немає `field_validator`?

<details>
<summary>Відповідь</summary>

Ні: Pydantic для `time` приймає лише час, а не дату з часом — `Input should be in a valid time format`. Тести парсера й моделі були зелені, бо кожен перевіряв свою половину на своїх даних. Помилку знайшов тест конвеєра; в уроці 41 модель бере час з ISO-рядка.
</details>

### Вправа 3

Напиши тест конвеєра: `demo_newsline.html` → `parse_rbc_news` → `validate_news`. Збережи `valid, rejected`; очікування — нічого не відхилено, а в першої новини `published_time` — `10:30`.

In [ ]:
from datetime import time

# YOUR CODE HERE
# BEGIN SOLUTION
valid, rejected = validate_news(parse_rbc_news(Path("tests/fixtures/demo_newsline.html").read_text(encoding="utf-8")))
# END SOLUTION

print([(item.title, item.published_time) for item in valid], "| відхилено:", len(rejected))
assert rejected == [] and valid[0].published_time == time(10, 30)
print("✅ Вправа 3 пройдена")

---
## 3. Мок мережі

`fake_session(html=..., error=...)` з `tests/unit/test_scraper.py` — замість `aiohttp.ClientSession`: `session.get(...)` віддає відповідь з цим HTML або кидає `error`.

### Вправа 4

Виклич `scraper.fetch_one` з фейковою сесією, що віддає вміст `rbc_newsline_item.html`. Результат — у `page, items`.

In [ ]:
import asyncio
from unittest import mock

import aiohttp

from news_hub import scraper
from tests.unit.test_scraper import URL, fake_session, http_error

feed = Path("tests/fixtures/rbc_newsline_item.html").read_text(encoding="utf-8")

# YOUR CODE HERE
# BEGIN SOLUTION
page, items = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
# END SOLUTION

print(page)
assert (page.count, page.error) == (1, None) and items[0]["datetime"] == "02:06"
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** `scraper.py` робить `from .parser import parse_rbc_news`. Тест патчить `news_hub.parser.parse_rbc_news`, щоб парсер повернув порожній список. Що поверне `fetch_one` для сторінки з вправи 4?

<details>
<summary>Відповідь</summary>

Одну новину — патч не подіяв. `fetch_one` шукає ім'я `parse_rbc_news` у модулі `scraper`, а патч замінив його лише в модулі `parser`. Patch where used: `news_hub.scraper.parse_rbc_news`.
</details>

### Вправа 5

Підміни парсер так, щоб `fetch_one` **справді** отримав порожній список: `mock.patch(..., return_value=[])` з правильним шляхом. Результат — у `page_patched`.

In [ ]:
with mock.patch("news_hub.parser.parse_rbc_news", return_value=[]):
    wrong_page, _ = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
print("patch news_hub.parser — новин:", wrong_page.count)

# YOUR CODE HERE
# BEGIN SOLUTION
with mock.patch("news_hub.scraper.parse_rbc_news", return_value=[]) as fake_parser:
    page_patched, _ = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
fake_parser.assert_called_once_with(feed)
# END SOLUTION

print("правильний patch — новин:", page_patched.count)
assert (wrong_page.count, page_patched.count) == (1, 0)
print("✅ Вправа 5 пройдена")

### Вправа 6

Мережа падає по-різному. Для кожної помилки зі списку `errors` виклич `fetch_one` з `fake_session(error=...)` і збери `page.error` у список `messages`. Жодна помилка не має вилетіти винятком.

In [ ]:
errors = [http_error(403), asyncio.TimeoutError(), aiohttp.ClientConnectionError("Connection reset by peer")]

# YOUR CODE HERE
# BEGIN SOLUTION
messages = []
for error in errors:
    page, _ = await scraper.fetch_one(fake_session(error=error), URL, t0=0.0)
    messages.append(page.error)
# END SOLUTION

for message in messages:
    print(message)
assert [m.split(":")[0] for m in messages] == ["ClientResponseError", "TimeoutError", "ClientConnectionError"]
print("✅ Вправа 6 пройдена")

---
## 4. Фейковий HTTP-сервер

Мок не виконує код між запитом і рядком HTML: з'єднання, статус, заголовки, декодування байтів. `aiohttp.test_utils.TestServer` — справжній сервер на `127.0.0.1` з вільним портом.

### Вправа 7

Додай до сервера маршрут `/forbidden/`, що відповідає `403`, і збери `scrape_all_async` з двох сторінок: `/feed/` і `/forbidden/`. Збережи `outcome`. Очікування: сторінка з `403` — з помилкою, новина з `/feed/` не загубилась.

In [ ]:
from urllib.parse import urlsplit

from aiohttp import web
from aiohttp.test_utils import TestServer


async def feed_page(request: web.Request) -> web.Response:
    return web.Response(text=feed, content_type="text/html")


site = web.Application()
site.router.add_get("/feed/", feed_page)

# YOUR CODE HERE
# BEGIN SOLUTION
async def forbidden(request: web.Request) -> web.Response:
    return web.Response(status=403, text="<h1>403 Forbidden</h1>", content_type="text/html")


site.router.add_get("/forbidden/", forbidden)
async with TestServer(site) as server:
    outcome = await scraper.scrape_all_async([str(server.make_url("/feed/")), str(server.make_url("/forbidden/"))])
# END SOLUTION

print([(urlsplit(p.url).path, p.count, (p.error or "")[:30]) for p in outcome.pages])
assert [p.count for p in outcome.pages] == [1, 0] and outcome.pages[1].error.startswith("ClientResponseError: 403")
assert len(outcome.news) == 1
print("✅ Вправа 7 пройдена")

---
## 5. API асинхронно

`httpx.AsyncClient(transport=httpx.ASGITransport(app=app))` викликає FastAPI напряму, в тому ж циклі подій. `lifespan` він не запускає — робимо це самі через `app.router.lifespan_context(app)`.

### Вправа 8

Надішли **одночасно** (`asyncio.gather`) 10 запитів `POST /api/scrape` з неправильним тілом `{"source": "ftp"}` і збережи відсортовані статуси в `statuses`. Скільки пройде до перевірки тіла (`422`), а скільки зупинить rate limit (`429`)?

In [ ]:
import httpx

from news_hub.api import app
from news_hub.middleware import RATE_LIMIT_REQUESTS

async with app.router.lifespan_context(app):
    await app.state.redis.flushdb()
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test") as aclient:
        print((await aclient.get("/health")).json())
        # YOUR CODE HERE
        # BEGIN SOLUTION
        responses = await asyncio.gather(*(aclient.post("/api/scrape", json={"source": "ftp"}) for _ in range(10)))
        statuses = sorted(r.status_code for r in responses)
        # END SOLUTION

print(statuses)
assert statuses == [422] * RATE_LIMIT_REQUESTS + [429] * (10 - RATE_LIMIT_REQUESTS)
print("✅ Вправа 8 пройдена")

**🔮 Прогноз:** Якщо замінити `RateLimiter` на «наївний» (`GET`, +1, `SET`), чи впаде цей тест на `fakeredis`?

<details>
<summary>Відповідь</summary>

Ні. `fakeredis` виконує команду, не віддаючи керування циклу подій, тож одночасні корутини не перемежовуються і гонки не буває. Атомарність доводить лише прогін на справжньому Redis (`TEST_REDIS_URL`) — там «наївний» лічильник пропускає всі 10.
</details>

---
## 6. Покриття

### Вправа 9

Запусти `run_pytest("-m", "unit", "--cov=news_hub", "--cov-report=term")` і знайди модуль з **найменшим** покриттям лише unit-тестами. Його ім'я (як у звіті, напр. `news_hub/api.py`) — у `lowest`. Чому саме він?

In [ ]:
report = run_pytest("-m", "unit", "--cov=news_hub", "--cov-report=term")
print("\n".join(line for line in report.splitlines() if line.startswith(("Name", "news_hub", "TOTAL"))))

# YOUR CODE HERE
# BEGIN SOLUTION
rows = [line.split() for line in report.splitlines() if line.startswith("news_hub/")]
percent = {row[0]: int(next(cell for cell in row if cell.endswith("%")).rstrip("%")) for row in rows}
lowest = min(percent, key=percent.get)
# END SOLUTION

print("найменше:", lowest)
assert lowest in {"news_hub/api.py", "news_hub/jobs.py", "news_hub/db.py", "news_hub/repository.py",
                  "news_hub/middleware.py", "news_hub/cache.py"}
print("✅ Вправа 9 пройдена — це код, що працює з базою чи Redis: його перевіряють інтеграційні тести")

---
## 7. Знайди помилку: «майже rbc.ua»

Так перевіряли адресу в уроках 36–39:

In [ ]:
def is_rbc_host_old(host: str | None) -> bool:
    return (host or "").endswith("rbc.ua")


for host in ("rbc.ua", "www.rbc.ua", "auto.rbc.ua"):
    assert is_rbc_host_old(host)
print("усі перевірки зелені")

Тести перевіряли лише те, що **має** пройти. Межових значень — «майже правильних» адрес — не було.

### Вправа 10

1. Додай до `hosts` адреси, які **не** мають пройти, і знайди серед них ту, яку `is_rbc_host_old` пропускає.
2. Напиши `is_rbc_host_fixed`: `rbc.ua` або піддомен `*.rbc.ua`.

In [ ]:
hosts = {"rbc.ua": True, "www.rbc.ua": True, "auto.rbc.ua": True}

# YOUR CODE HERE
# BEGIN SOLUTION
hosts |= {"fakerbc.ua": False, "rbc.ua.evil.com": False, "": False}


def is_rbc_host_fixed(host: str | None) -> bool:
    return host == "rbc.ua" or (host or "").endswith(".rbc.ua")
# END SOLUTION

missed = [host for host, allowed in hosts.items() if is_rbc_host_old(host) != allowed]
print("стара перевірка помиляється на:", missed)
assert missed, "додай адресу, на якій стара перевірка помиляється"
assert all(is_rbc_host_fixed(host) == allowed for host, allowed in hosts.items())
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Що перевіряє тест конвеєра, чого не перевіряють тести парсера й моделі окремо?
2. Де патчити `parse_rbc_news`, щоб підмінити його для `fetch_one`, і чому?
3. Яку помилку бачить фейковий сервер, але не бачить мок?
4. Чим `httpx.AsyncClient` + `ASGITransport` зручніший за `TestClient`, а в чому незручніший?
5. Чому 100% покриття — не мета?

## Далі

- Книга курсу: [Урок 41](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_41/) — чотири помилки, які знайшли тести, покрокова діаграма «patch where used», межа фейку, покриття до/після.
- У папці `news_hub`: `pytest --cov=news_hub`; на справжніх серверах — `docker compose up -d db redis` і `TEST_DATABASE_URL=… TEST_REDIS_URL=redis://localhost:6379/15 pytest`.
- Урок 42 — AI-інструменти розробника: тести — перший спосіб перевірити згенерований код.